In [181]:
import json
import argparse
import pandas as pd
from datetime import datetime
from google.cloud import bigquery, storage
from flight_prices.transform import transform_flights

In [182]:
parser = argparse.ArgumentParser()
parser.add_argument ('--date', default = datetime.today().strftime('%Y-%m-%d'))

args, unknown = parser.parse_known_args()

date = args.date


In [183]:
BUCKET_NAME = "flights-pipeline-data-lake-ya"
PREFIX = f"raw/flights/extraction_date={date}/"

In [184]:
client_gcs = storage.Client()
client_bq = bigquery.Client()

In [185]:
#This stores a list of blob objects in the path, not the actual json contents.
blobs = [
    blob
    for blob in client_gcs.list_blobs(BUCKET_NAME, prefix=PREFIX)
    if blob.name.endswith(".json")
]

In [186]:
#download as text downloads json to memory and json.loads reads json as a python object.
#dfs = [] creates an empty list to store your transformed DataFrames.
#The loop downloads each JSON file into memory.
#json.loads() converts each JSON string into a Python dictionary.
#transform_flights(data) transforms each dictionary into a DataFrame.
#dfs.append(df) adds each transformed DataFrame to your list.
#pd.concat() combines everything into one DataFrame.

dfs = []

for blob in blobs:
    raw_json = blob.download_as_text()
    data = json.loads(raw_json)

    df = transform_flights(data, blob.name)
    dfs.append(df)

df_final = pd.concat(dfs, ignore_index = True)

In [201]:
df_final.head()

,observation_id,extraction_date,departure_airport,arrival_airport,outbound_date,return_date,total_duration,stops,price,currency,type,flights,source_file
0,ad1fb265697f80249764b7e6f05035c0bf4700b2f66d06...,2026-10-16,BHX,MRU,2027-01-07,2027-01-16,925,1,1618,GBP,Round trip,"[{""airline"": ""Emirates"", ""airline_logo"": ""http...",BHX_MRU_20261016_124126_890956.json
1,92ac3031314e15d210d28170411e9e214c7e8cd1f9b7e1...,2026-10-16,BHX,MRU,2027-01-07,2027-01-16,980,1,1622,GBP,Round trip,"[{""airline"": ""Emirates"", ""airline_logo"": ""http...",BHX_MRU_20261016_124126_890956.json
2,b12116a482195f79da4a57f37e7f48d2d3a4e3b63f2d7d...,2026-10-16,BHX,MRU,2027-01-07,2027-01-16,1315,2,1782,GBP,Round trip,"[{""airline"": ""Lufthansa"", ""airline_logo"": ""htt...",BHX_MRU_20261016_124126_890956.json
3,50a16149c1513100a95344dab5d53dc409ac9df154b130...,2026-10-16,BHX,MRU,2027-01-07,2027-01-16,950,1,1958,GBP,Round trip,"[{""airline"": ""Turkish Airlines"", ""airline_logo...",BHX_MRU_20261016_124126_890956.json
4,3aea519305332ec132254d1e90523473e8b6a337190224...,2026-10-16,BHX,MRU,2027-01-07,2027-01-16,955,1,2007,GBP,Round trip,"[{""airline"": ""Air France"", ""airline_logo"": ""ht...",BHX_MRU_20261016_124126_890956.json


In [ ]:
#ensure dataframe is not empty
if df_final.empty:
    raise ValueError("No flight observations to process.")

In [208]:
#check for duplicate observation IDs or null entries
if (
    not df_final["observation_id"].duplicated().any()
    and not df_final["observation_id"].isna().any()
):
    print("Validation passed. Proceeding with staging.")
else:
    raise ValueError("Duplicate or null observation IDs found.")

Validation passed. Proceeding with staging.


In [202]:
#set table id
table_id = "flights-pipeline-510915.flights_warehouse.staging_flights"

In [203]:
#create schema for the BigQuery staging table
staging_schema = [
    bigquery.SchemaField("observation_id", "STRING"),
    bigquery.SchemaField("extraction_date", "DATE"),
    bigquery.SchemaField("departure_airport", "STRING"),
    bigquery.SchemaField("arrival_airport", "STRING"),
    bigquery.SchemaField("outbound_date", "DATE"),
    bigquery.SchemaField("return_date", "DATE"),
    bigquery.SchemaField("total_duration", "INTEGER"),
    bigquery.SchemaField("stops", "INTEGER"),
    bigquery.SchemaField("price", "INTEGER"),
    bigquery.SchemaField("currency", "STRING"),
    bigquery.SchemaField("type", "STRING"),
    bigquery.SchemaField("flights", "STRING"),
    bigquery.SchemaField("source_file", "STRING"),
  
]

In [204]:
job_config = bigquery.LoadJobConfig(schema=staging_schema, write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        create_disposition=bigquery.CreateDisposition.CREATE_IF_NEEDED)

In [205]:
load_job = client_bq.load_table_from_dataframe(
    dataframe = df_final,
    destination = table_id,
    job_config=job_config)
load_job.result()

print(f"Successfully loaded {load_job.output_rows} rows into staging.")


/Users/Yasin/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Successfully loaded 53 rows into staging.


In [209]:
merge_sql = """
MERGE `flights-pipeline-510915.flights_warehouse.final_flights` AS target

USING `flights-pipeline-510915.flights_warehouse.staging_flights` AS source

ON target.observation_id = source.observation_id

WHEN NOT MATCHED THEN
    INSERT (
        observation_id,
        extraction_date,
        departure_airport,
        arrival_airport,
        outbound_date,
        return_date,
        total_duration,
        stops,
        price,
        currency,
        type,
        flights,
        source_file
    )
    VALUES (
        source.observation_id,
        source.extraction_date,
        source.departure_airport,
        source.arrival_airport,
        source.outbound_date,
        source.return_date,
        source.total_duration,
        source.stops,
        source.price,
        source.currency,
        source.type,
        source.flights,
        source.source_file
    )
"""

In [211]:
merge_job = client_bq.query(merge_sql)

merge_job.result()

print(f"MERGE completed: {merge_job.num_dml_affected_rows} rows inserted.")

MERGE completed: 0 rows inserted.
